In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
"""
train.py
--------
Fine-tune một model Llama nhỏ (miễn phí) bằng LoRA, dùng dữ liệu train.jsonl
đã được build sẵn từ build_train_dataset.py

Yêu cầu cài đặt (chạy 1 lần):
    pip install transformers trl peft accelerate bitsandbytes datasets torch

Cách chạy:
    python train.py

Sau khi train xong, model LoRA sẽ được lưu vào thư mục ./output_model
"""

'\ntrain.py\n--------\nFine-tune một model Llama nhỏ (miễn phí) bằng LoRA, dùng dữ liệu train.jsonl\nđã được build sẵn từ build_train_dataset.py\n\nYêu cầu cài đặt (chạy 1 lần):\n    pip install transformers trl peft accelerate bitsandbytes datasets torch\n\nCách chạy:\n    python train.py\n\nSau khi train xong, model LoRA sẽ được lưu vào thư mục ./output_model\n'

In [16]:
!pip install transformers trl peft accelerate bitsandbytes datasets torch

In [17]:
!pip install -U torchao -q

In [18]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

In [19]:
# ============================================================
# 1. CẤU HÌNH
# ============================================================

# Model nhỏ, free, phổ biến cho fine-tune trên máy yếu / Colab free
# Có thể đổi sang các model nhỏ khác nếu muốn, ví dụ:
#   - "meta-llama/Llama-3.2-1B-Instruct"
#   - "meta-llama/Llama-3.2-3B-Instruct"
#   - "Qwen/Qwen2.5-1.5B-Instruct" (không cần xin quyền truy cập)
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

TRAIN_FILE = "/content/drive/MyDrive/MockProject_NguyenTuanPhat/Day 12/output/train.jsonl"
OUTPUT_DIR = "/content/drive/MyDrive/MockProject_NguyenTuanPhat/Day 12/output_model"

# Tự động phát hiện có GPU (CUDA) hay không
USE_GPU = torch.cuda.is_available()

In [20]:
# ============================================================
# 2. LOAD MODEL
#    - Có GPU  -> dùng 4-bit quantization (nhẹ VRAM, nhanh)
#    - Không GPU -> load bình thường, train bằng CPU (chậm hơn nhưng vẫn chạy được)
# ============================================================

def load_model_and_tokenizer():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    if USE_GPU:
        print("Phát hiện GPU -> load model ở chế độ 4-bit")
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            quantization_config=bnb_config,
            device_map="auto",
        )
        model = prepare_model_for_kbit_training(model)
    else:
        print("Không có GPU -> load model ở CPU (float32, sẽ train chậm hơn)")
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.float32,
            device_map={"": "cpu"},
        )

    return model, tokenizer

In [21]:
# ============================================================
# 3. GẮN LORA VÀO MODEL (chỉ train 1 phần nhỏ tham số, rất nhẹ)
# ============================================================

def apply_lora(model):
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )
    return get_peft_model(model, lora_config)


In [22]:
# ============================================================
# 4. LOAD DATASET (train.jsonl) VÀ ÁP DỤNG CHAT TEMPLATE
# ============================================================

def load_and_format_dataset(tokenizer):
    if not os.path.exists(TRAIN_FILE):
        raise FileNotFoundError(
            f"Không tìm thấy {TRAIN_FILE}. "
            f"Hãy chạy build_train_dataset.py trước để tạo file này."
        )

    dataset = load_dataset("json", data_files=TRAIN_FILE, split="train")

    def format_sample(sample):
        text = tokenizer.apply_chat_template(
            sample["messages"],
            tokenize=False,
            add_generation_prompt=False,
        )
        return {"text": text}

    dataset = dataset.map(format_sample)
    return dataset

In [25]:
# ============================================================
# 5. TRAIN
# ============================================================

def main():
    print("=" * 50)
    print("Loading model + tokenizer...")
    model, tokenizer = load_model_and_tokenizer()

    print("Applying LoRA...")
    model = apply_lora(model)

    print("Loading dataset...")
    dataset = load_and_format_dataset(tokenizer)
    print(f"Total training samples: {len(dataset)}")

    training_args = SFTConfig(
        output_dir=OUTPUT_DIR,
        num_train_epochs=1,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        logging_steps=10,
        save_strategy="epoch",
        fp16=False,          # fp16 chỉ dùng được khi có GPU
        use_cpu=not USE_GPU,
        report_to="none",
        dataset_text_field="text",
        max_length=1024,
    )

    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=dataset,
    )

    print("Bắt đầu train...")
    trainer.train()

    print(f"Train xong. Lưu model vào {OUTPUT_DIR}")
    trainer.save_model(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)

    print("=" * 50)
    print("HOÀN TẤT")
    print("=" * 50)

In [26]:
if __name__ == "__main__":
    main()

Loading model + tokenizer...
Phát hiện GPU -> load model ở chế độ 4-bit


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Applying LoRA...
Loading dataset...
Total training samples: 300


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Bắt đầu train...


Step,Training Loss
10,1.371806
20,1.074803
30,1.036577


Train xong. Lưu model vào /content/drive/MyDrive/MockProject_NguyenTuanPhat/Day 12/output_model
HOÀN TẤT
